<a href="https://colab.research.google.com/github/HleBBondar/amazon-sales-analytics-ab-testing/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Amazon_Sale_Report.csv", on_bad_lines='skip')

df_cleaned = df.drop_duplicates() # Dropping duplicates

df_cleaned = df_cleaned.dropna(axis=0, how='all') # Dropping empty lines

df_cleaned = df_cleaned.drop(columns = ['index', 'currency','Sales Channel ', 'fulfilled-by','Unnamed: 22', 'ship-country'] )
#Dropping all unnecessary columns

# Checking what values should be rewritten, capitalized or filled in
"""columns = [ 'Status', 'Fulfilment', 'Sales Channel ',
       'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN',
       'Courier Status', 'Qty', 'currency', 'ship-city',
       'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids',
       'B2B', 'fulfilled-by']
for column in df_cleaned:
  if column in columns:
     print(df_cleaned[column].unique()) """
#Changing everything to string
to_str = ['Status', 'Fulfilment', 'ship-service-level',
          'Style', 'SKU', 'Category', 'Size', 'ASIN',
          'ship-state', 'ship-city','Order ID']
for column in to_str:
  df_cleaned[column] = df_cleaned[column].astype(str).str.strip()

# Capitalizing everything we need to capitalize
to_capitalize = ['Category','ship-city','ship-state'  ]
for column in to_capitalize:
  df_cleaned[column] = df_cleaned[column].str.title()

#print(df_cleaned['Status'].unique()) - cheking if removing any statuses is needed
# Doing categorical consolidation
status_replace = {
    # Grouping into 'Cancelled'
    'Shipped - Returned to Seller': 'Cancelled',
    'Shipped - Rejected by Buyer': 'Cancelled',
    'Shipped - Damaged': 'Cancelled',
    'Shipped - Lost in Transit': 'Cancelled',
    'Shipped - Returning to Seller': 'Cancelled',

    # Grouping into 'Shipped'
    'Shipped - Picked Up': 'Shipped',

    # Grouping into 'Pending'
    'Pending - Waiting for Pick Up': 'Pending',
    'Shipping': 'Pending'
}
df_cleaned['Status'] = df_cleaned['Status'].replace(status_replace)

#Changing mistakes in region
state_replace = {'Rajshthan':'Rajasthan',
                 'Pb':'Punjab',
                 'Ar': 'Arunachal Pradesh',
                 'Rj': 'Rajasthan',
                 'Nl':'New Delhi',
                 'Punjab/Mohali/Zirakpur' : 'Punjab',
                 'Pondicherry':'Puducherry',
                 'Orissa':'Odisha', }

df_cleaned['ship-state'] = df_cleaned['ship-state'].replace(state_replace)
#Adding date: month, year, day of the week and weekend status
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'], format = 'mixed')
df_cleaned["Year"]= df_cleaned['Date'].dt.year
df_cleaned["Month"] = df_cleaned['Date'].dt.month_name()
df_cleaned["Day"] = df_cleaned['Date'].dt.day_name()
df_cleaned["Is Weekend"] = df_cleaned['Day'].isin(['Saturday', 'Sunday'])
#print(df_cleaned['Courier Status'].unique()) - cheking if filling in the gaps is possible
df_cleaned['Courier Status'] = df_cleaned['Courier Status'].fillna('Cancelled') #filled all nan

"""Regarding the Amount column, crucially, revenue should not be counted for
 Cancelled orders since no actual income was generated. Furthermore, the dataset
 contained an anomaly where certain rows recorded a non-zero Amount but a Qty of 0,
 indicating a system data-entry error. To protect overall profitability metrics from artificial inflation,
  the Amount in both of these scenarios was adjusted to zero."""
cancelled_or_zero_qty = (df_cleaned['Status'] == 'Cancelled') | (df_cleaned['Qty'] == 0)
df_cleaned.loc[cancelled_or_zero_qty, 'Amount'] = 0
df_cleaned['promotion-ids'] = df_cleaned['promotion-ids'].fillna('No promotion') #filled all nan
#print(df_cleaned['promotion-ids'].unique())
df_cleaned['ship-postal-code'] = df_cleaned['ship-postal-code'].fillna('Unknown') #filled all nan
df_cleaned['Amount'] = df_cleaned['Amount'].fillna(0) #filled all nan
#print(df_cleaned['ship-postal-code'].unique())
df_cleaned.to_csv('cleaned_data.csv', index=False)
print(df_cleaned['Month'].unique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_5163/3814111952.py:6: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Amazon_Sale_Report.csv", on_bad_lines='skip')


['April' 'March' 'May' 'June']
